In [19]:
import os
import json
import re
import pdfplumber
import pytesseract
import cv2
import numpy as np
from PIL import Image
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

pytesseract.pytesseract.tesseract_cmd = r"C:\Users\pkeer\Downloads\tesseract-ocr-w64-setup-5.5.0.20241111.exe"


In [14]:
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text


In [3]:
def extract_text_from_image(img_path):
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    text = pytesseract.image_to_string(gray)
    return text


In [4]:
def get_text(file_path):
    if file_path.endswith(".pdf"):
        return extract_text_from_pdf(file_path)
    elif file_path.endswith((".png", ".jpg", ".jpeg")):
        return extract_text_from_image(file_path)
    elif file_path.endswith(".txt"):
        return open(file_path, "r", encoding="utf-8").read()
    else:
        return ""


In [5]:
# --- EXTRACTION PROMPT ---
# This is the "brain" of the extraction process.
# The LLM converts messy OCR text into strict structured JSON.

EXTRACTION_PROMPT = """You are a medical document information extraction system.

You are given the FULL OCR text of a diagnostic lab report.
Extract structured information and return ONLY valid JSON in the exact schema below.

====================
REQUIRED OUTPUT SCHEMA
====================
{
  "patient_information": {
    "patient_name": string | null,
    "age_years": number | null,
    "gender": string | null,
    "lab_number": string | null,
    "report_status": string | null,
    "collection_datetime": string | null,
    "report_datetime": string | null,
    "laboratory_name": string | null
  },

  "tests_index": {
    "<canonical_test_key>": {
      "test_name": string,
      "value": number | string | null,
      "units": string | null,
      "reference_range": string | null,
      "interpretation": "low" | "normal" | "high" | "borderline" | null,
      "category": string
    }
  },

  "tests_by_category": {
    "complete_blood_count": [ "<canonical_test_key>" ],
    "liver_function": [ "<canonical_test_key>" ],
    "kidney_function": [ "<canonical_test_key>" ],
    "lipid_profile": [ "<canonical_test_key>" ],
    "thyroid_profile": [ "<canonical_test_key>" ],
    "diabetes_related": [ "<canonical_test_key>" ],
    "vitamins_and_minerals": [ "<canonical_test_key>" ],
    "electrolytes": [ "<canonical_test_key>" ],
    "other_tests": [ "<canonical_test_key>" ]
  },

  "abnormal_findings": [
    {
      "canonical_test_key": string,
      "observed_value": number | string,
      "expected_range": string,
      "severity": "low" | "high" | "critical"
    }
  ],

  "clinical_notes": {
    "interpretations": [ string ],
    "comments": [ string ],
    "notes": [ string ],
    "recommendations": [ string ],
    "disclaimers": [ string ]
  },

  "metadata": {
    "total_pages": number,
    "page_numbers_detected": [ number ],
    "report_end_marker_present": boolean
  }
}

====================
CANONICAL TEST KEY RULES
====================
- Each test MUST appear exactly ONCE in tests_index
- Use normalized snake_case keys
- Remove symbols, punctuation, and methods
- Use same canonical key everywhere

====================
INTERPRETATION RULES
====================
- Use reference range to detect low/normal/high
- If borderline mentioned → borderline
- Otherwise → normal

====================
STRICT RULES
====================
- Return ONLY valid JSON
- No explanation
- No markdown
- No comments
- No duplicate tests
- Missing values → null
"""



def extract_medical_json(ocr_text):

    full_prompt = EXTRACTION_PROMPT + "\n\nOCR TEXT:\n" + ocr_text

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Return ONLY valid JSON. No text."},
            {"role": "user", "content": full_prompt}
        ],
        temperature=0,
        max_tokens=8000
    )

    return response.choices[0].message.content

In [15]:
import gradio as gr
import json
import re
import os

# -------- JSON CLEANER --------
def clean_json(text):
    text = re.sub(r"```json|```", "", text).strip()
    start = text.find("{")
    end = text.rfind("}")
    json_str = text[start:end+1]
    json_str = re.sub(r",\s*}", "}", json_str)
    json_str = re.sub(r",\s*]", "]", json_str)
    return json.loads(json_str)


# -------- MAIN PROCESS FUNCTION --------
def process_file(file):

    if file is None:
        return "No file uploaded"

    file_path = file.name

    # Extract OCR text
    ocr_text = get_text(file_path)
    print("OCR TEXT LENGTH:", len(ocr_text))
    print(ocr_text[:500])   # preview first 500 chars

    if not ocr_text.strip():
        return "No text extracted (check OCR/Tesseract)"

    # Call LLM
    json_output = extract_medical_json(ocr_text)

    # Clean JSON
    try:
        parsed = clean_json(json_output)
    except:
        return "Invalid JSON returned by model:\n\n" + json_output

    # Save JSON
    with open("output.json", "w") as f:
        json.dump(parsed, f, indent=2)

    return json.dumps(parsed, indent=2)


# -------- GRADIO UI --------
demo = gr.Interface(
    fn=process_file,
    inputs=gr.File(label="Upload Medical Report (PDF/Image/TXT)"),
    outputs=gr.Textbox(label="Extracted JSON"),
    title="Medical Report → Structured JSON Extractor",
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


OCR TEXT LENGTH: 20124
.
Name : DUMMY
Lab No. : 439854467 Age : 30 Years
Ref By : U Gender : Male
Collected : 14/5/2023 11:03:00AM Reported : 16/5/2023 1:36:25PM
A/c Status : P Report Status : Final
Collected at : L P L-ROHINI (NATIONAL REFERENCE LAB) Processed at : L P L-NATIONAL REFERENCE LAB
National Reference laboratory, Block E, Sector National Reference laboratory, Block E,
18, ROHINI Sector 18, Rohini, New Delhi -110085
DELHI 110085
Test Report
Test Name Results Units Bio. Ref. Interval
SwasthFit Super 4
COMPLE


In [16]:
def process_file(file):
    try:
        if file is None:
            return "No file uploaded"

        file_path = file.name
        print("Processing:", file_path)

        # -------- OCR --------
        ocr_text = get_text(file_path)

        print("OCR TEXT LENGTH:", len(ocr_text))
        print("OCR PREVIEW:\n", ocr_text[:500])

        if not ocr_text.strip():
            return "OCR failed or no text extracted."

        # -------- LLM --------
        json_output = extract_medical_json(ocr_text)

        # ===== PASTE STEP-3 HERE =====
        print("\nRAW MODEL OUTPUT:\n")
        print(json_output[:1000])
        # ============================

        # -------- CLEAN JSON --------
        parsed_json = clean_json(json_output)

        # -------- SAVE JSON --------
        with open("output.json", "w", encoding="utf-8") as f:
            json.dump(parsed_json, f, indent=2)

        print("Saved → output.json")

        return json.dumps(parsed_json, indent=2)

    except Exception as e:
        import traceback
        print(traceback.format_exc())
        return f"ERROR:\n{str(e)}"